# 02 — Text Normalization & Preprocessing Pipeline

**Module:** `src/preprocess.py`

Real-world business names are messy: accented characters, legal suffixes (Pvt Ltd, LLC), social handles (@brand), domain names (brand.com), zero-padded street numbers, and fake-null placeholders ("N/A", "null"). A robust preprocessing pipeline normalizes these variations so that two records referring to the same business produce similar strings — which makes downstream string similarity features more reliable.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from src.preprocess import preprocess_business_name, preprocess_address, strip_accents, is_non_latin

print("Preprocessing module loaded successfully.")

### 1. Accent Stripping (Latin-safe, Indic-preserving)

`strip_accents()` removes diacritics from Latin characters (é→e, ü→u) while **preserving** Indic scripts (Devanagari, Bengali, etc.) that use combining marks as integral parts of the character.

In [ ]:
examples_accent = [
    'Café Mumbai Pvt Ltd',
    'Moore Bítwise Inc',
    'B+ Rétail Incorporated',
    'राम मार्केटिंग प्राइवेट लिमिटेड',
]

rows = []
for raw in examples_accent:
    cleaned = strip_accents(raw)
    rows.append({'Raw Input': raw, 'After strip_accents()': cleaned,
                 'Changed?': '✓' if raw != cleaned else '—'})

pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

### 2. Social Handle & Domain Removal

Business listings sometimes contain Twitter/Instagram handles or website URLs instead of proper names. `preprocess_business_name()` strips these so they don't pollute the string features.

In [ ]:
examples_social = [
    '@primemoney',
    'moorebitwise.com',
    'dhaninvestments.in',
    '@theshop_delhi',
]

rows = []
for raw in examples_social:
    cleaned = preprocess_business_name(raw)
    rows.append({'Raw Input': raw, 'After preprocess_business_name()': cleaned,
                 'Changed?': '✓' if raw != cleaned else '—'})

pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

### 3. Legal Suffix Removal

"Pvt Ltd", "Private Limited", "LLC", "Corp", "Holdings" are **informationally redundant** in entity matching — they appear in both records and cancel out. Removing them reduces noise and improves Jaccard / Jaro-Winkler scores on the meaningful tokens.

In [ ]:
examples_legal = [
    'Sharma General Stores Pvt Ltd',
    'Dhan Investments Private Limited',
    'ABC Corp LLC',
    'XYZ Holdings Group',
]

rows = []
for raw in examples_legal:
    cleaned = preprocess_business_name(raw)
    rows.append({'Raw Input': raw, 'After preprocess_business_name()': cleaned,
                 'Changed?': '✓' if raw != cleaned else '—'})

pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

### 4. Fake-Null / Placeholder Scrubbing

Data pipelines sometimes leave sentinel strings instead of actual NULL. We detect and normalize these so downstream code can handle them as proper missing values.

In [ ]:
examples_null = ['N/A', 'null', 'None', '--', 'undefined']

rows = []
for raw in examples_null:
    cleaned = preprocess_business_name(raw)
    rows.append({'Raw Input': raw, 'After preprocess_business_name()': repr(cleaned),
                 'Changed?': '✓' if raw != cleaned else '—'})

pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

### 5. Address Zero-Padding Rectification

Street numbers are sometimes stored with leading zeros (e.g., `0017560 ELLIS ROAD`). `preprocess_address()` strips these so that `17560 ELLIS ROAD` matches `17560 ELLIS ROAD`.

In [ ]:
examples_addr = [
    '0017560 ELLIS ROAD',
    '0337 Oakland Ave',
    'PHOENIX, AZ, 1712 MONTEBELLO AVE',
]

rows = []
for raw in examples_addr:
    cleaned = preprocess_address(raw)
    rows.append({'Raw Address': raw, 'After preprocess_address()': cleaned,
                 'Changed?': '✓' if raw != cleaned else '—'})

pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

### 6. Full Pipeline on Real Pairs from Dataset

Here we apply the full preprocessing pipeline to actual records from the benchmark dataset and show the before/after transformation side-by-side.

In [ ]:
pairs = pd.read_csv('data/processed/benchmark_pairs.csv')
sample = pairs.head(10)[['s1_name', 'probe_name', 'pair_type', 'label']].copy()

sample['s1_name_clean']    = sample['s1_name'].apply(preprocess_business_name)
sample['probe_name_clean'] = sample['probe_name'].apply(preprocess_business_name)

display_df = sample[['s1_name', 's1_name_clean', 'probe_name', 'probe_name_clean',
                      'pair_type', 'label']]
display_df.style.set_properties(**{'text-align': 'left', 'font-size': '12px'})